# 🏦 Multimodal RAG Financial — Chạy trên Google Colab với Ollama

Notebook này hướng dẫn chạy toàn bộ hệ thống **Multimodal RAG Financial** trên Google Colab:

| Bước | Nội dung |
|------|----------|
| 1 | Kiểm tra GPU và cài Ollama |
| 2 | Pull model `qwen2.5:3b` |
| 3 | Clone repo và cài dependencies |
| 4 | Cấu hình môi trường |
| 5 | Demo: Nạp báo cáo mẫu |
| 6 | Demo: Hỏi đáp có Ollama |
| 7 | Khám phá Knowledge Graph |
| 8 | Nạp PDF tài chính thực tế |
| 9 | Khởi chạy Web UI ngay trong Colab |

> ⚠️ **Chọn Runtime → Change runtime type → GPU (T4)** để Ollama chạy nhanh hơn.
> Nếu không có GPU, notebook vẫn chạy được trên CPU nhưng sinh câu trả lời sẽ chậm hơn.

---
## 1. Kiểm tra môi trường và cài Ollama

In [ ]:
# Kiểm tra GPU
!nvidia-smi || echo '⚠️ Không phát hiện GPU. Ollama sẽ chạy trên CPU (chậm hơn).'

In [ ]:
# Cài đặt Ollama trên Linux (Colab)
!curl -fsSL https://ollama.com/install.sh | sh
print('✅ Ollama đã cài xong.')

In [ ]:
# Khởi động Ollama server ở chế độ nền
import subprocess, time, urllib.request, os

# Kiểm tra nếu Ollama đã đang chạy sẵn
try:
    urllib.request.urlopen('http://127.0.0.1:11434/api/tags', timeout=2)
    print('✅ Ollama server đã đang chạy sẵn!')
except Exception:
    ollama_log = open('/tmp/ollama.log', 'w')
    ollama_proc = subprocess.Popen(
        ['ollama', 'serve'],
        stdout=ollama_log,
        stderr=subprocess.STDOUT,
    )
    print(f'Đang khởi động Ollama server (PID: {ollama_proc.pid})...')

    # Chờ server sẵn sàng (tối đa 30s)
    for i in range(30):
        try:
            urllib.request.urlopen('http://127.0.0.1:11434/api/tags', timeout=2)
            print('✅ Ollama server đã sẵn sàng!')
            break
        except Exception:
            time.sleep(1)
    else:
        print('❌ Ollama server không khởi động được. Log chi tiết:')
        if os.path.exists('/tmp/ollama.log'):
            with open('/tmp/ollama.log', 'r') as f:
                print(f.read())


---
## 2. Tải model Ollama

Mặc định dùng `qwen2.5:3b` (~2 GB). Bạn có thể đổi sang model khác nếu muốn.

In [ ]:
OLLAMA_MODEL = 'qwen2.5:3b'  # Đổi model tại đây nếu muốn

!ollama pull {OLLAMA_MODEL}
print(f'\n✅ Model {OLLAMA_MODEL} đã sẵn sàng.')

# Xác nhận model đã tải
!ollama list

---
## 3. Clone repo và cài dependencies

In [ ]:
import os

REPO_URL = 'https://github.com/dongdx2k5/Financial-Multimodal-RAG.git'
REPO_DIR = '/content/Financial-Multimodal-RAG'

# Hỗ trợ cả trường hợp đã upload thư mục zip giải nén sẵn
if os.path.exists('/content/Financial-Multimodal-RAG-main'):
    REPO_DIR = '/content/Financial-Multimodal-RAG-main'
elif not os.path.exists(REPO_DIR):
    !git clone --branch main {REPO_URL} {REPO_DIR}
    print('✅ Clone xong.')
else:
    !cd {REPO_DIR} && git pull
    print('✅ Đã cập nhật repo.')

os.chdir(REPO_DIR)
print(f'📂 Thư mục hiện tại: {os.getcwd()}')


In [ ]:
# Cài đặt project và dependencies
!pip install -e '.[dev]' -q
print('\n✅ Đã cài xong tất cả dependencies.')

---
## 4. Cấu hình môi trường

Tạo file `.env` cho Colab. Ollama server chạy tại `127.0.0.1:11434`.

In [ ]:
env_content = f"""OPENAI_API_KEY=
LLM_PROVIDER=ollama
OLLAMA_BASE_URL=http://127.0.0.1:11434
OLLAMA_MODEL={OLLAMA_MODEL}
OLLAMA_TIMEOUT=180
OCR_ENABLED=false
OCR_LANGUAGE=vie+eng
"""

with open('.env', 'w') as f:
    f.write(env_content)

print('✅ File .env đã được tạo:')
!cat .env

In [ ]:
# Kiểm tra kết nối Ollama từ Python
import sys
sys.path.insert(0, 'src')

from financial_rag.config import Settings
from financial_rag.ollama_client import probe

settings = Settings()
status = probe(settings)
print(f"Provider : {status['provider']}")
print(f"Model    : {status['model']}")
print(f"Ready    : {status['ready']}")
print(f"Message  : {status['message']}")

---
## 5. Demo: Nạp báo cáo mẫu

Nạp file `samples/sample_report.txt` để kiểm tra pipeline hoạt động:
- Đọc văn bản → Chunking → Trích xuất facts → Kiểm tra chất lượng → Xây dựng graph.

In [ ]:
from financial_rag.service import FinancialRAGService
from financial_rag.config import Settings

settings = Settings()
service = FinancialRAGService(settings)

# Nạp báo cáo mẫu
result = service.ingest('samples/sample_report.txt', ticker='DEMO', year=2024)

print('📊 Kết quả nạp tài liệu:')
print(f"  Document ID  : {result['document_id']}")
print(f"  Số trang     : {result['pages']}")
print(f"  Số chunks    : {result['chunks']}")
print(f"  Số facts     : {result['facts']}")
print(f"  Số bảng      : {result['tables']}")
print(f"  Phạm vi      : {result['scope']}")
if result.get('warnings'):
    print(f"  ⚠️ Cảnh báo  : {result['warnings']}")
if result.get('checks'):
    print(f"  🔍 Kiểm tra  : {len(result['checks'])} phép kiểm công thức")
    for check in result['checks']:
        print(f"     {check['formula']} → {check['status']}")

---
## 6. Demo: Hỏi đáp với Ollama

Thử hỏi các câu hỏi tài chính. Hệ thống sẽ:
1. Tìm facts có cấu trúc trước (mode `structured`/`calculation`)
2. Nếu cần diễn giải → gọi Ollama (mode `llm`)
3. Nếu Ollama không đạt kiểm tra trích dẫn → trả bằng chứng gốc (mode `evidence`)

In [ ]:
def ask_and_display(question, use_llm=True):
    """Hỏi và hiển thị kết quả."""
    print(f'\n{"="*60}')
    print(f'❓ Câu hỏi: {question}')
    print(f'{"="*60}')

    response = service.ask(question, top_k=5, use_llm=use_llm)

    print(f'📌 Mode: {response.mode}')
    if response.llm_model:
        print(f'🤖 Model: {response.llm_model}', end='')
        if response.llm_elapsed_seconds:
            print(f' ({response.llm_elapsed_seconds}s)', end='')
        print()
    print(f'\n💬 Trả lời:\n{response.answer}')

    if response.evidence:
        print(f'\n📎 Bằng chứng ({len(response.evidence)}):')
        for i, ev in enumerate(response.evidence, 1):
            text = ev.text[:120] + '...' if len(ev.text) > 120 else ev.text
            print(f'  [{i}] Trang {ev.page}: {text}')
            if ev.graph_facts:
                for fact in ev.graph_facts:
                    print(f'      → {fact.metric}: {fact.raw_value} {fact.unit or ""} '
                          f'(kỳ {fact.period_label}, chất lượng: {fact.quality})')

    if response.warnings:
        print(f'\n⚠️ Cảnh báo: {response.warnings}')

In [ ]:
# --- Câu hỏi số liệu (trả lời trực tiếp từ facts, không cần LLM) ---
ask_and_display('Doanh thu thuần năm 2024 là bao nhiêu?', use_llm=False)

In [ ]:
# --- Câu hỏi có gọi Ollama để diễn giải ---
ask_and_display('Tình hình tài chính của DEMO năm 2024 như thế nào?', use_llm=True)

In [ ]:
# --- Câu hỏi khác ---
ask_and_display('Lợi nhuận sau thuế năm 2024?', use_llm=True)

In [ ]:
# --- Hỏi tự do: Thay đổi câu hỏi bên dưới ---
MY_QUESTION = 'Tổng tài sản và nợ phải trả năm 2024?'  # ← Sửa tại đây

ask_and_display(MY_QUESTION, use_llm=True)

---
## 7. Khám phá Knowledge Graph

Xem cấu trúc đồ thị tri thức đã xây dựng từ báo cáo.

In [ ]:
graph = service.graph.graph

print('🔗 Knowledge Graph:')
print(f'  Số node: {graph.number_of_nodes()}')
print(f'  Số edge: {graph.number_of_edges()}')

# Đếm theo loại node
from collections import Counter
kinds = Counter(attrs.get('kind') for _, attrs in graph.nodes(data=True))
print(f'\n📊 Phân bố node:')
for kind, count in kinds.most_common():
    kind_str = str(kind or 'Khác')
    print(f'  {kind_str:10s}: {count}')

# Đếm theo loại quan hệ
relations = Counter(attrs.get('relation') for _, _, attrs in graph.edges(data=True))
print(f'\n🔗 Phân bố quan hệ:')
for rel, count in relations.most_common():
    rel_str = str(rel or 'Khác')
    print(f'  {rel_str:15s}: {count}')


In [ ]:
# Hiển thị các Facts đã trích xuất
print('📋 Facts đã trích xuất:')
print(f'{"":-<80}')
for node_id, attrs in graph.nodes(data=True):
    if attrs.get('kind') == 'Fact':
        metric_str = str(attrs.get('metric') or '?')[:35]
        val_str = str(attrs.get('raw_value') or '?')
        unit_str = str(attrs.get('unit') or '')
        period_str = str(attrs.get('period_label') or attrs.get('period') or '?')
        quality_str = str(attrs.get('quality') or '?')
        print(f"  {metric_str:35s} | {val_str:>15s} {unit_str:4s} | Kỳ: {period_str:12s} | Chất lượng: {quality_str}")


In [ ]:
# Vẽ sơ đồ đơn giản của graph (nếu có matplotlib)
try:
    import matplotlib.pyplot as plt
    import networkx as nx

    fig, ax = plt.subplots(1, 1, figsize=(14, 8))

    color_map = {
        'Company': '#FF6B6B', 'Report': '#4ECDC4', 'Period': '#45B7D1',
        'Chunk': '#96CEB4', 'Fact': '#FFEAA7', 'Metric': '#DDA0DD'
    }
    node_colors = [color_map.get(graph.nodes[n].get('kind', ''), '#CCCCCC') for n in graph.nodes()]

    labels = {}
    for n, attrs in graph.nodes(data=True):
        kind = attrs.get('kind', '')
        if kind == 'Company':
            labels[n] = str(attrs.get('ticker', n))
        elif kind == 'Report':
            labels[n] = f"Report\n{str(attrs.get('name', ''))[:20]}"
        elif kind == 'Period':
            labels[n] = f"Năm {attrs.get('year', '?')}"
        elif kind == 'Metric':
            labels[n] = str(attrs.get('name', n))[:25]
        elif kind == 'Fact':
            labels[n] = f"{attrs.get('raw_value', '?')}\n{str(attrs.get('unit') or '')}"
        elif kind == 'Chunk':
            labels[n] = f"Chunk\np.{attrs.get('page', '?')}"
        else:
            labels[n] = str(n)[:15]

    pos = nx.spring_layout(graph, k=2.5, seed=42)
    nx.draw_networkx_nodes(graph, pos, ax=ax, node_color=node_colors,
                           node_size=800, alpha=0.9)
    nx.draw_networkx_labels(graph, pos, ax=ax, labels=labels, font_size=7)
    nx.draw_networkx_edges(graph, pos, ax=ax, arrows=True,
                           edge_color='#888888', alpha=0.5, arrowsize=12)

    edge_labels = {(u, v): d.get('relation', '') for u, v, d in graph.edges(data=True)}
    nx.draw_networkx_edge_labels(graph, pos, ax=ax, edge_labels=edge_labels,
                                  font_size=6, font_color='#555555')

    # Legend
    for kind, color in color_map.items():
        ax.scatter([], [], c=color, s=100, label=kind)
    ax.legend(loc='upper left', fontsize=9)
    ax.set_title('Knowledge Graph — Multimodal RAG Financial', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()
except Exception as e:
    print(f'⚠️ Không vẽ được đồ thị ({e}). Chạy "!pip install matplotlib" nếu cần.')


---
## 8. Nạp PDF tài chính thực tế

Upload file PDF báo cáo tài chính của bạn. Hệ thống sẽ:
- Đọc từng trang, nhận diện bảng mã chỉ tiêu
- Trích xuất số liệu có cấu trúc (facts)
- Kiểm tra công thức kế toán
- Xây dựng đồ thị tri thức

In [ ]:
from google.colab import files

print('📎 Chọn file PDF/TXT/HTML báo cáo tài chính để upload...')
uploaded = files.upload()

if uploaded:
    for filename, content in uploaded.items():
        filepath = f'/content/{filename}'
        with open(filepath, 'wb') as f:
            f.write(content)
        print(f'\n📄 Đang nạp: {filename} ({len(content):,} bytes)...')

        # Nhập ticker và năm (tùy chọn)
        ticker = input('Nhập mã cổ phiếu (Enter để bỏ qua): ').strip() or None
        year_str = input('Nhập năm báo cáo (Enter để bỏ qua): ').strip()
        try:
            year = int(year_str) if year_str else None
        except ValueError:
            year = None
            print(f'⚠️ Năm không phải số nguyên ({year_str}), đã bỏ qua.')

        result = service.ingest(filepath, ticker=ticker, year=year)

        print(f'\n✅ Nạp xong: {filename}')
        print(f"  Document ID     : {result['document_id']}")
        print(f"  Số trang        : {result['pages']}")
        print(f"  Chunks          : {result['chunks']}")
        print(f"  Facts trích xuất: {result['facts']}")
        print(f"  Dòng bảng chuẩn : {result['statement_rows']}")
        print(f"  Trang OCR       : {result['ocr_pages']}")
        print(f"  Phạm vi         : {result['scope']}")

        if result.get('checks'):
            print(f'\n🔍 Kiểm tra công thức kế toán ({len(result["checks"])}):' )
            for check in result['checks']:
                icon = '✅' if check['status'] == 'pass' else ('❌' if check['status'] == 'fail' else '⚠️')
                print(f"  {icon} {check['formula']} → {check['status']}")
                if check.get('difference') and check['status'] == 'fail':
                    print(f"     Chênh lệch: {check['difference']}")

        if result.get('warnings'):
            print(f'\n⚠️ Cảnh báo ({len(result["warnings"])}):')
            for w in result['warnings']:
                print(f'  • {w}')
else:
    print('Không có file nào được upload.')


In [ ]:
# --- Hỏi về PDF vừa nạp ---
# Liệt kê các báo cáo đã nạp
print('📚 Các báo cáo đã nạp:')
for node_id, attrs in service.graph.graph.nodes(data=True):
    if attrs.get('kind') == 'Report':
        name_str = str(attrs.get('name') or '?')[:40]
        ticker_str = str(attrs.get('ticker') or '-')[:6]
        year_str = str(attrs.get('year') or '-')
        scope_str = str(attrs.get('scope') or '-')
        print(f"  • {name_str:40s} | ID: {node_id} | Ticker: {ticker_str:6s} | Năm: {year_str:4s} | Scope: {scope_str}")


In [ ]:
# Hỏi về tất cả các báo cáo
ask_and_display('Doanh thu thuần là bao nhiêu?', use_llm=True)

In [ ]:
# Hỏi về một báo cáo cụ thể (copy document_id ở ô trên)
DOC_ID = None  # ← Dán document_id vào đây, ví dụ: 'abc123...'

if DOC_ID:
    response = service.ask('Lợi nhuận sau thuế?', top_k=5,
                           document_id=DOC_ID, use_llm=True)
    print(f'Mode: {response.mode}')
    print(f'Trả lời: {response.answer}')
else:
    print('Hãy gán DOC_ID = document_id của báo cáo cần hỏi.')

---
## 9. Khởi chạy Web UI ngay trong Colab

Dùng **Colab built-in** để mở giao diện web — **không cần đăng ký, không cần token**.
Chỉ cần chạy 2 cell là xong.

In [ ]:
import subprocess, time, os, urllib.request

os.chdir(REPO_DIR)

# Kiểm tra nếu FastAPI server đã đang chạy sẵn
try:
    urllib.request.urlopen('http://127.0.0.1:8000/health', timeout=2)
    print('✅ FastAPI server đã đang chạy sẵn!')
except Exception:
    uvicorn_log = open('/tmp/uvicorn.log', 'w')
    server_proc = subprocess.Popen(
        ['python', '-m', 'uvicorn', 'financial_rag.api:app',
         '--host', '127.0.0.1', '--port', '8000'],
        cwd=REPO_DIR,
        env={**os.environ, 'PYTHONPATH': f'{REPO_DIR}/src'},
        stdout=uvicorn_log,
        stderr=subprocess.STDOUT,
    )
    print(f'Đang khởi động FastAPI server (PID: {server_proc.pid})...')

    # Chờ server sẵn sàng (tối đa 15s)
    for i in range(15):
        try:
            urllib.request.urlopen('http://127.0.0.1:8000/health', timeout=2)
            print('✅ FastAPI server đã sẵn sàng!')
            break
        except Exception:
            time.sleep(1)
    else:
        print('⚠️ Server chưa sẵn sàng. Kiểm tra log:')
        if os.path.exists('/tmp/uvicorn.log'):
            with open('/tmp/uvicorn.log', 'r') as f:
                print(f.read())


In [ ]:
# Mở Web UI ngay trong notebook (iframe nhúng)
from google.colab import output

print('🌐 Web UI hiển thị bên dưới:')
print('   Nếu iframe bị chặn, chạy cell tiếp theo để mở trong tab mới.\n')
output.serve_kernel_port_as_iframe(8000, height=700)

In [ ]:
# Mở trong tab trình duyệt mới (nếu iframe ở trên không hoạt động)
from google.colab import output

print('🔗 Nhấn link bên dưới để mở Web UI trong tab mới:')
output.serve_kernel_port_as_window(8000)

In [ ]:
# Kiểm tra health của toàn hệ thống
import json, urllib.request

with urllib.request.urlopen('http://127.0.0.1:8000/health') as r:
    health = json.load(r)
print('🏥 Health check:')
for k, v in health.items():
    print(f'  {k:15s}: {v}')

print()
with urllib.request.urlopen('http://127.0.0.1:8000/health/ollama') as r:
    ollama_health = json.load(r)
print('🤖 Ollama health:')
for k, v in ollama_health.items():
    print(f'  {k:15s}: {v}')

---
## 🛑 Dọn dẹp

Chạy cell dưới khi muốn dừng tất cả server.

In [ ]:
# Dừng tất cả server
try:
    if 'server_proc' in locals() and server_proc:
        server_proc.terminate()
        print('✅ FastAPI server đã dừng.')
except Exception:
    pass

try:
    if 'ollama_proc' in locals() and ollama_proc:
        ollama_proc.terminate()
        print('✅ Ollama server đã dừng.')
except Exception:
    pass

print('\n🏁 Đã dọn dẹp xong.')
